# ML4Sci GSoC 2026 — DeepLense Evaluation Tests
**Applicant:** Hondi Birindwa Prisca  
**GitHub:** https://github.com/Prisca-Nova  
**Email:** pbirindw@andrew.cmu.edu  
**Project of interest:** DEEPLENSE6 — Gravitational Lens Finding  

This notebook covers:
- **Common Test I:** Multi-class classification of simulated lensing images (3 classes)
- **Specific Test V:** Binary lens-finding on real imbalanced observational data

Both implemented in PyTorch with full evaluation metrics (ROC, AUC).

---
## Setup & Imports

In [ ]:
# Install dependencies if needed
# !pip install torch torchvision scikit-learn matplotlib seaborn tqdm numpy

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, roc_auc_score, classification_report
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

---
# COMMON TEST I — Multi-Class Gravitational Lens Classification

**Task:** Classify simulated lensing images into 3 classes:
- `no_sub` — no substructure
- `subhalo` — subhalo substructure  
- `vortex` — vortex substructure

**Strategy:**  
I use a pre-trained **EfficientNet-B0** fine-tuned on the lensing dataset. Key choices:
1. **Transfer learning** from ImageNet — strong visual feature prior, efficient on small datasets.
2. **Custom augmentation** — random flips, rotations, and Gaussian noise to capture rotational symmetry of lensing images and improve generalization.
3. **Label smoothing** in the loss to prevent overconfidence.
4. **Cosine annealing LR schedule** for stable convergence.
5. **90:10 stratified train/val split** as required.

### 1.1 — Data Loading (Common Test I)

> **Before running:** Download `dataset.zip` from the provided Google Drive link and extract it.  
> Expected structure:
> ```
> dataset/
>   no_sub/     *.npy files
>   subhalo/    *.npy files
>   vortex/     *.npy files
> ```

In [ ]:
# ── Dataset path — update this to where you extracted dataset.zip ──
DATASET_PATH_T1 = './dataset'   # <-- update if needed

CLASS_NAMES_T1 = ['no_sub', 'subhalo', 'vortex']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES_T1)}

def load_npy_dataset(root):
    """Load all .npy files from class subdirectories."""
    images, labels = [], []
    for cls in CLASS_NAMES_T1:
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir):
            # Try flat structure with class prefix
            cls_dir = root
        files = [f for f in os.listdir(cls_dir) if cls in f and f.endswith('.npy')]
        # If all files are in root directory with class name in filename
        if not files and os.path.isdir(os.path.join(root, cls)):
            files = [f for f in os.listdir(os.path.join(root, cls)) if f.endswith('.npy')]
            for f in files:
                arr = np.load(os.path.join(root, cls, f))
                images.append(arr)
                labels.append(CLASS_TO_IDX[cls])
        else:
            # flat npy with shape (N, H, W) or (N, C, H, W)
            npy_files = [f for f in os.listdir(cls_dir) if f.endswith('.npy') and cls in f]
            for f in npy_files:
                arr = np.load(os.path.join(cls_dir, f))
                if arr.ndim == 3:  # (N, H, W)
                    for img in arr:
                        images.append(img)
                        labels.append(CLASS_TO_IDX[cls])
                elif arr.ndim == 2:  # single (H, W)
                    images.append(arr)
                    labels.append(CLASS_TO_IDX[cls])
    return images, labels


class LensDatasetT1(Dataset):
    """
    Dataset for Common Test I.
    Handles both (N,H,W) numpy arrays per class and single-image .npy files.
    Images are already min-max normalized; we apply additional augmentations.
    """
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx].astype(np.float32)
        # Ensure shape is (H, W) then expand to (3, H, W) for EfficientNet
        if img.ndim == 2:
            img = np.stack([img, img, img], axis=0)
        elif img.ndim == 3 and img.shape[0] == 1:
            img = np.repeat(img, 3, axis=0)
        elif img.ndim == 3 and img.shape[-1] in (1, 3):
            img = np.transpose(img, (2, 0, 1))
            if img.shape[0] == 1:
                img = np.repeat(img, 3, axis=0)
        img = torch.tensor(img)
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


# ── Transforms ─────────────────────────────────────────────────────
# Lensing images have rotational symmetry — aggressive rotation augmentation is valid
train_transform_t1 = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=180),   # full rotation valid for lensing
    transforms.Resize((64, 64)),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

val_transform_t1 = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

print('Dataset loading functions defined.')
print('Update DATASET_PATH_T1 and run the next cell to load data.')

In [ ]:
# ── Load and split data ─────────────────────────────────────────────
# Adapt this cell based on the actual structure of your dataset.zip

# Option A: .npy files, one per class, shape (N, H, W)
all_images, all_labels = [], []
for cls in CLASS_NAMES_T1:
    path = os.path.join(DATASET_PATH_T1, f'{cls}.npy')
    if os.path.exists(path):
        arr = np.load(path)        # (N, H, W)
        for img in arr:
            all_images.append(img)
            all_labels.append(CLASS_TO_IDX[cls])
        print(f'Loaded {len(arr)} images for class "{cls}"')
    else:
        # Option B: subdirectory of .npy per image
        cls_dir = os.path.join(DATASET_PATH_T1, cls)
        if os.path.isdir(cls_dir):
            files = [f for f in os.listdir(cls_dir) if f.endswith('.npy')]
            for f in files:
                img = np.load(os.path.join(cls_dir, f))
                all_images.append(img)
                all_labels.append(CLASS_TO_IDX[cls])
            print(f'Loaded {len(files)} images for class "{cls}" from subdirectory')

print(f'\nTotal samples: {len(all_labels)}')
print(f'Class distribution: { {c: all_labels.count(i) for i,c in enumerate(CLASS_NAMES_T1)} }')

# 90:10 stratified split
idx = list(range(len(all_labels)))
train_idx, val_idx = train_test_split(idx, test_size=0.1, stratify=all_labels, random_state=SEED)

train_imgs = [all_images[i] for i in train_idx]
train_lbls = [all_labels[i] for i in train_idx]
val_imgs   = [all_images[i] for i in val_idx]
val_lbls   = [all_labels[i] for i in val_idx]

print(f'Train: {len(train_lbls)} | Val: {len(val_lbls)}')

train_ds_t1 = LensDatasetT1(train_imgs, train_lbls, transform=train_transform_t1)
val_ds_t1   = LensDatasetT1(val_imgs,   val_lbls,   transform=val_transform_t1)

train_loader_t1 = DataLoader(train_ds_t1, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader_t1   = DataLoader(val_ds_t1,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print('DataLoaders ready.')

### 1.2 — EDA: Visualize Sample Images

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for row, cls in enumerate(CLASS_NAMES_T1):
    cls_imgs = [all_images[i] for i, l in enumerate(all_labels) if l == row][:4]
    for col, img in enumerate(cls_imgs):
        ax = axes[row][col]
        ax.imshow(img if img.ndim == 2 else img[0], cmap='hot', origin='lower')
        ax.set_title(f'{cls}' if col == 0 else '', fontsize=10, fontweight='bold')
        ax.axis('off')
plt.suptitle('Common Test I — Sample Images per Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_images_t1.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.3 — Model: EfficientNet-B0 with Fine-Tuning

In [ ]:
def build_efficientnet_t1(num_classes=3, pretrained=True):
    """
    EfficientNet-B0 fine-tuned for lens classification.
    Strategy:
    - Load ImageNet pretrained weights for strong visual feature prior.
    - Freeze first ~60% of layers to preserve low-level features.
    - Replace classifier head for 3-class output.
    - Add Dropout(0.3) before final layer to regularize.
    """
    weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = models.efficientnet_b0(weights=weights)

    # Freeze early layers
    params = list(model.parameters())
    freeze_until = int(len(params) * 0.6)
    for i, p in enumerate(params):
        if i < freeze_until:
            p.requires_grad = False

    # Replace classifier
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model


model_t1 = build_efficientnet_t1(num_classes=3).to(DEVICE)

trainable = sum(p.numel() for p in model_t1.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_t1.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

### 1.4 — Training

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=20,
                lr=1e-3, label_smoothing=0.1, device=DEVICE):
    """
    Training loop with:
    - CrossEntropyLoss + label smoothing (reduces overconfidence)
    - AdamW optimizer (weight decay for regularization)
    - CosineAnnealingLR schedule
    - Best model checkpointing on val loss
    """
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val_loss = float('inf')
    best_state = None

    for epoch in range(num_epochs):
        # ── Train ──
        model.train()
        running_loss = 0.0
        for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}', leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)

        # ── Validate ──
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out = model(imgs)
                val_loss += criterion(out, labels).item() * imgs.size(0)
                preds = out.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        val_loss   = val_loss / len(val_loader.dataset)
        val_acc    = correct / total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        scheduler.step()

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | '
                  f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

    # Restore best
    model.load_state_dict(best_state)
    print(f'\nBest val loss: {best_val_loss:.4f}')
    return model, history


model_t1, history_t1 = train_model(model_t1, train_loader_t1, val_loader_t1, num_epochs=25)

In [ ]:
# Save weights
torch.save(model_t1.state_dict(), 'model_t1_efficientnet.pth')
print('Model saved.')

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history_t1['train_loss']) + 1)

ax1.plot(epochs, history_t1['train_loss'], label='Train Loss')
ax1.plot(epochs, history_t1['val_loss'],   label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss'); ax1.legend()

ax2.plot(epochs, history_t1['val_acc'], color='green', label='Val Accuracy')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy'); ax2.legend()

plt.tight_layout()
plt.savefig('training_curves_t1.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.5 — Evaluation: ROC Curves & AUC (Common Test I)

In [ ]:
def evaluate_multiclass(model, loader, class_names, device=DEVICE):
    """Compute ROC curves and AUC for multi-class using one-vs-rest strategy."""
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_labels.extend(labels.numpy())

    all_probs  = np.vstack(all_probs)   # (N, num_classes)
    all_labels = np.array(all_labels)
    preds      = all_probs.argmax(axis=1)

    # Binarize labels for one-vs-rest ROC
    labels_bin = label_binarize(all_labels, classes=list(range(len(class_names))))

    # Per-class ROC
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(len(class_names)):
        fpr[i], tpr[i], _ = roc_curve(labels_bin[:, i], all_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Macro average
    macro_auc = roc_auc_score(labels_bin, all_probs, average='macro', multi_class='ovr')

    # ── Plot ROC ──
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, (cls, color) in enumerate(zip(class_names, colors)):
        ax.plot(fpr[i], tpr[i], color=color, lw=2,
                label=f'{cls} (AUC = {roc_auc[i]:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'Common Test I — ROC Curves (Macro AUC = {macro_auc:.4f})')
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig('roc_t1.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\n=== Classification Report ===')
    print(classification_report(all_labels, preds, target_names=class_names))
    print(f'\nPer-class AUC:')
    for i, cls in enumerate(class_names):
        print(f'  {cls:12s}: {roc_auc[i]:.4f}')
    print(f'  Macro AUC  : {macro_auc:.4f}')

    return all_probs, all_labels, macro_auc


probs_t1, labels_t1, auc_t1 = evaluate_multiclass(model_t1, val_loader_t1, CLASS_NAMES_T1)

---
# SPECIFIC TEST V — Gravitational Lens Finding on Real Imbalanced Data

**Task:** Binary classification — lens vs. non-lens — on real HSC observational images.  
**Key challenge:** Severe class imbalance (non-lenses >> lenses).

**Strategy:**
1. **WeightedRandomSampler** to oversample the minority class (lenses) during training — ensures the model sees balanced batches.
2. **Focal Loss** instead of standard BCE — down-weights easy negatives, focuses learning on hard examples (critical for imbalanced detection tasks, directly analogous to my fraud detection research).
3. **Multi-channel input** — images have 3 filters (g, r, i bands), giving the model color/spectral information.
4. **EfficientNet-B0** fine-tuned for binary output with sigmoid activation.
5. Evaluation on the provided `test_lenses` / `test_nonlenses` directories.

### 2.1 — Data Loading (Test V)

> **Before running:** Download and extract the Test V dataset.  
> Expected structure:
> ```
> dataset_testV/
>   train_lenses/     *.npy files  (shape: 3, 64, 64 per image)
>   train_nonlenses/  *.npy files
>   test_lenses/      *.npy files
>   test_nonlenses/   *.npy files
> ```

In [ ]:
DATASET_PATH_T5 = './dataset_testV'   # <-- update if needed


class LensDatasetT5(Dataset):
    """
    Dataset for Test V. Each .npy file is shape (3, 64, 64) — 3 photometric bands.
    Label: 1 = lens, 0 = non-lens.
    """
    def __init__(self, root_dir, split='train', transform=None):
        self.transform = transform
        self.data = []
        self.labels = []

        for cls_name, label in [('lenses', 1), ('nonlenses', 0)]:
            folder = os.path.join(root_dir, f'{split}_{cls_name}')
            if not os.path.isdir(folder):
                print(f'WARNING: {folder} not found')
                continue
            files = [f for f in os.listdir(folder) if f.endswith('.npy')]
            for f in files:
                arr = np.load(os.path.join(folder, f)).astype(np.float32)  # (3, 64, 64)
                self.data.append(arr)
                self.labels.append(label)

        print(f'{split}: {self.labels.count(1)} lenses, {self.labels.count(0)} non-lenses '
              f'(ratio 1:{self.labels.count(0)//max(1,self.labels.count(1))})')

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = torch.tensor(self.data[idx])  # (3, 64, 64)
        # Per-channel min-max normalization (images may not be pre-normalized)
        for c in range(img.shape[0]):
            mn, mx = img[c].min(), img[c].max()
            if mx > mn:
                img[c] = (img[c] - mn) / (mx - mn)
        if self.transform:
            img = self.transform(img)
        return img, float(self.labels[idx])


# Transforms — same rotational logic, plus stronger augmentation for rare class
train_transform_t5 = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=180),
    transforms.Resize((64, 64)),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])
val_transform_t5 = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_ds_t5 = LensDatasetT5(DATASET_PATH_T5, split='train', transform=train_transform_t5)
test_ds_t5  = LensDatasetT5(DATASET_PATH_T5, split='test',  transform=val_transform_t5)

In [ ]:
# ── WeightedRandomSampler for class imbalance ──────────────────────
# Gives each sample a weight inversely proportional to its class frequency.
# Result: model sees ~equal batches of lenses and non-lenses.

train_labels_t5 = train_ds_t5.labels
class_counts = [train_labels_t5.count(0), train_labels_t5.count(1)]
class_weights = [1.0 / c for c in class_counts]
sample_weights = [class_weights[l] for l in train_labels_t5]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader_t5 = DataLoader(train_ds_t5, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
test_loader_t5  = DataLoader(test_ds_t5,  batch_size=32, shuffle=False,   num_workers=2, pin_memory=True)

print(f'Class weights → non-lens: {class_weights[0]:.5f} | lens: {class_weights[1]:.5f}')
print('DataLoaders ready.')

### 2.2 — Focal Loss

Focal Loss down-weights the loss contribution from easy, well-classified negatives (non-lenses),  
directing the gradient signal toward hard-to-classify positives (rare lenses).  
This is my primary tool for handling extreme class imbalance, same approach I use in my mobile money fraud detection research.

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss for binary classification.
    FL(p) = -alpha * (1-p)^gamma * log(p)
    
    gamma > 0 reduces the relative loss for well-classified examples.
    alpha balances positive vs negative class importance.
    """
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none')
        p_t = torch.exp(-bce)
        focal = self.alpha * (1 - p_t) ** self.gamma * bce
        return focal.mean() if self.reduction == 'mean' else focal.sum()


print('FocalLoss defined (alpha=0.25, gamma=2.0).')

### 2.3 — Model for Test V

In [ ]:
def build_efficientnet_t5(pretrained=True):
    """
    EfficientNet-B0 for binary lens detection.
    Outputs a single logit (sigmoid applied during evaluation).
    """
    weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = models.efficientnet_b0(weights=weights)

    # Freeze early layers, fine-tune from block 5 onward
    params = list(model.parameters())
    freeze_until = int(len(params) * 0.5)
    for i, p in enumerate(params):
        if i < freeze_until:
            p.requires_grad = False

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, 1)  # binary output
    )
    return model


model_t5 = build_efficientnet_t5().to(DEVICE)
trainable = sum(p.numel() for p in model_t5.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,}')

### 2.4 — Training (Test V)

In [ ]:
def train_binary(model, train_loader, val_loader, num_epochs=25,
                 lr=5e-4, device=DEVICE):
    criterion = FocalLoss(alpha=0.25, gamma=2.0)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    best_auc  = 0.0
    best_state = None

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}', leave=False):
            imgs   = imgs.to(device)
            labels = labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)

        # Validate
        model.eval()
        val_loss = 0.0
        v_probs, v_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs   = imgs.to(device)
                labels_t = labels.to(device).unsqueeze(1)
                out = model(imgs)
                val_loss += criterion(out, labels_t).item() * imgs.size(0)
                probs = torch.sigmoid(out).cpu().numpy().squeeze()
                v_probs.extend(probs if probs.ndim > 0 else [probs.item()])
                v_labels.extend(labels.numpy())

        train_loss /= len(train_loader.dataset)
        val_loss   /= len(val_loader.dataset)
        val_auc     = roc_auc_score(v_labels, v_probs)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auc'].append(val_auc)

        if val_auc > best_auc:
            best_auc   = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        scheduler.step()

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | '
                  f'Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f}')

    model.load_state_dict(best_state)
    print(f'\nBest Val AUC: {best_auc:.4f}')
    return model, history


model_t5, history_t5 = train_binary(model_t5, train_loader_t5, test_loader_t5, num_epochs=30)

In [ ]:
torch.save(model_t5.state_dict(), 'model_t5_lens_finder.pth')
print('Model saved.')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history_t5['train_loss']) + 1)
ax1.plot(epochs, history_t5['train_loss'], label='Train Loss')
ax1.plot(epochs, history_t5['val_loss'],   label='Val Loss')
ax1.set_title('Test V — Loss Curves'); ax1.legend()
ax2.plot(epochs, history_t5['val_auc'], color='green')
ax2.set_title('Test V — Val AUC'); ax2.set_ylabel('AUC')
plt.tight_layout()
plt.savefig('training_curves_t5.png', dpi=150)
plt.show()

### 2.5 — Final Evaluation on Test Set (Test V)

In [ ]:
def evaluate_binary(model, loader, device=DEVICE):
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            probs  = torch.sigmoid(logits).cpu().numpy().squeeze()
            all_probs.extend(probs if probs.ndim > 0 else [probs.item()])
            all_labels.extend(labels.numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    test_auc = auc(fpr, tpr)

    # Youden's J statistic: best threshold
    j_scores = tpr - fpr
    best_thresh = thresholds[np.argmax(j_scores)]
    preds = (all_probs >= best_thresh).astype(int)

    # ROC plot
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, color='#1f77b4', lw=2, label=f'ROC (AUC = {test_auc:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.scatter(fpr[np.argmax(j_scores)], tpr[np.argmax(j_scores)],
               color='red', s=80, zorder=5, label=f'Best threshold = {best_thresh:.3f}')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title('Specific Test V — Lens Finding ROC Curve')
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig('roc_t5.png', dpi=150)
    plt.show()

    print(f'Test AUC   : {test_auc:.4f}')
    print(f'Best thresh: {best_thresh:.4f} (Youden\'s J)')
    print('\n=== Classification Report ===')
    print(classification_report(all_labels, preds, target_names=['non-lens', 'lens']))

    return all_probs, all_labels, test_auc


probs_t5, labels_t5, auc_t5 = evaluate_binary(model_t5, test_loader_t5)

### 2.6 — Visual Inspection: Top Lens Candidates

In [ ]:
# Show the images the model is most confident are lenses
sorted_idx = np.argsort(probs_t5)[::-1]
top_lens   = sorted_idx[:8]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()
for ax, idx in zip(axes, top_lens):
    # Get image from dataset
    img_arr, lbl = test_ds_t5[idx]
    # Show g-band (channel 0)
    ax.imshow(img_arr[0].numpy(), cmap='hot', origin='lower')
    true_lbl = 'Lens' if lbl == 1.0 else 'Non-lens'
    ax.set_title(f'P={probs_t5[idx]:.3f} | True: {true_lbl}', fontsize=9)
    ax.axis('off')
plt.suptitle('Top 8 Model-Predicted Lens Candidates (g-band)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('top_candidates_t5.png', dpi=150)
plt.show()

---
## Summary of Results

In [ ]:
print('='*55)
print('  ML4Sci GSoC 2026 — DeepLense Evaluation Summary')
print('='*55)
print(f'  Common Test I  (multi-class, EfficientNet-B0)')
print(f'    Macro AUC (val)   : {auc_t1:.4f}')
print()
print(f'  Specific Test V (lens finding, imbalanced data)')
print(f'    AUC (test set)    : {auc_t5:.4f}')
print()
print('  Key design decisions:')
print('    - EfficientNet-B0 pretrained, partial fine-tuning')
print('    - Rotational augmentation (lensing symmetry)')
print('    - WeightedRandomSampler (class imbalance)')
print('    - Focal Loss, gamma=2 (hard example mining)')
print('    - Youden J threshold selection')
print('='*55)